In [1]:
import random
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import xgboost as xgb
import mlflow
import optuna
import datetime

from sklearn.model_selection import (
    train_test_split,
)
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import roc_auc_score, f1_score

/Users/hmnguyen1067/Downloads/Github/Bank-Customer-Churn-Prediction-Pipeline/.pixi/envs/default/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
seed = 42
random.seed(seed)
np.random.seed(seed)

mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment("Bank-Customer-Churn-Prediction-Experiment")

2025/09/15 15:03:42 INFO mlflow.tracking.fluent: Experiment with name 'Bank-Customer-Churn-Prediction-Experiment' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/1', creation_time=1757923422892, experiment_id='1', last_update_time=1757923422892, lifecycle_stage='active', name='Bank-Customer-Churn-Prediction-Experiment', tags={}>

# Data preprocessing

In [3]:
data_path = "../data/Customer-Churn-Records.csv"


def clean_data(data_path):
    df = pd.read_csv(data_path)
    df.head()

    return df


def split_data(df):
    # Train/val/test stratified split of ratio 0.8/0.1/0.1
    labels = df.Exited.values
    del df["Exited"]

    X_train, X_vtest, y_train, y_vtest = train_test_split(
        df, labels, test_size=0.2, random_state=seed, stratify=labels
    )

    X_val, X_test, y_val, y_test = train_test_split(
        X_vtest, y_vtest, test_size=0.5, random_state=seed, stratify=y_vtest
    )

    return X_train, y_train, X_val, y_val, X_test, y_test

In [4]:
cat = [
    "Geography",
    "Gender",
    "NumOfProducts",
    "HasCrCard",
    "IsActiveMember",
    "Satisfaction Score",
    "Card Type",
]

num = ["CreditScore", "Age", "Tenure", "Balance", "EstimatedSalary", "Point Earned"]


def preprocess_data(X_train, X_val, X_test):
    preprocessor = ColumnTransformer(
        [
            ("oh", OneHotEncoder(handle_unknown="ignore"), cat),
            ("scaler", StandardScaler(), num),
        ]
    )

    X_train = preprocessor.fit_transform(X_train)
    X_val = preprocessor.transform(X_val)
    X_test = preprocessor.transform(X_test)

    with mlflow.start_run():
        mlflow.sklearn.log_model(preprocessor, "ChurnDataPreprocessor")

    return X_train, X_val, X_test, preprocessor

In [5]:
records = clean_data(data_path)
X_train, y_train, X_val, y_val, X_test, y_test = split_data(records)
X_train_tf, X_val_tf, X_test_tf, pp = preprocess_data(X_train, X_val, X_test)

2025/09/15 15:03:43 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2025/09/15 15:03:43 WARNING mlflow.sklearn: Model was missing function: predict. Not logging python_function flavor!
2025/09/15 15:03:44 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
2025/09/15 15:03:44 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


🏃 View run persistent-colt-754 at: http://localhost:5000/#/experiments/1/runs/ec094ea004644cde985bff9df08cc7ca
🧪 View experiment at: http://localhost:5000/#/experiments/1


# Model Evaluation and Hyperparameters Tuning

In [6]:
sampler = optuna.samplers.TPESampler(seed=seed)

In [7]:
def xgb_objective(trial):
    with mlflow.start_run(nested=True):
        train = xgb.DMatrix(X_train_tf, label=y_train)
        valid = xgb.DMatrix(X_val_tf, label=y_val)

        params = {
            "n_estimators": trial.suggest_int("n_estimators", 100, 5000),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 1.0, log=True),
            "reg_lambda": trial.suggest_float("reg_lambda", 1e-9, 100.0, log=True),
            "reg_alpha": trial.suggest_float("reg_alpha", 1e-9, 100.0, log=True),
            "subsample": trial.suggest_float("subsample", 0.1, 1.0),
            "max_depth": trial.suggest_int("max_depth", 1, 12),
            "max_delta_step": trial.suggest_int("max_delta_step", 0, 10),
            "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
            "gamma": trial.suggest_float("gamma", 1e-9, 0.5, log=True),
            "scale_pos_weight": trial.suggest_float(
                "scale_pos_weight", 1e-6, 500.0, log=True
            ),
            "seed": seed,
        }

        model = xgb.train(
            params,
            train,
            evals=[(valid, "validation")],
            early_stopping_rounds=300,
            verbose_eval=False,
        )

        preds = model.predict(valid)
        pred_labels = np.clip(np.rint(preds), 0, 1)

        f1 = f1_score(y_val, pred_labels)
        roc_auc = roc_auc_score(y_val, pred_labels)

        mlflow.log_metric("roc_auc", roc_auc)
        mlflow.log_metric("f1_score", f1)
        mlflow.log_params(params)

    return roc_auc


def train_best_xgb_model(X_train, y_train, best_params):
    train = xgb.DMatrix(X_train, label=y_train)

    model = xgb.train(best_params, train)

    return model


def plot_feature_importance(model, feat_names=None):
    """
    Plots feature importance for an XGBoost model.

    Args:
    - model: A trained XGBoost model

    Returns:
    - fig: The matplotlib figure object
    """
    fig, ax = plt.subplots(figsize=(10, 8))
    importance_type = "gain"
    if feat_names is not None:
        model.feature_names = list(feat_names)

    xgb.plot_importance(
        model,
        importance_type=importance_type,
        ax=ax,
        title=f"Feature Importance based on {importance_type}",
    )
    plt.tight_layout()
    plt.close(fig)

    return fig


def hyperparameter_tuning(X_train, y_train, feat_names=None):
    with mlflow.start_run(
        run_name=f"xgboost_hyperparameter_tuning_{datetime.datetime.now().date()}",
        nested=True,
    ):
        mlflow.set_tag("model", "xgboost")
        study_xgb = optuna.create_study(direction="maximize", sampler=sampler)
        study_xgb.optimize(xgb_objective, n_trials=200)

        print("Number of finished trials:", len(study_xgb.trials))
        print("Best value:", study_xgb.best_value)

        mlflow.log_params(study_xgb.best_params)

        xgb_model = train_best_xgb_model(X_train, y_train, study_xgb.best_params)

        mlflow.xgboost.log_model(
            xgb_model=xgb_model,
            name="mlflow_model",
            input_example=X_train[:5],
            registered_model_name="XGBoostChurnModel",
        )

        importances = plot_feature_importance(
            xgb_model,
            feat_names=feat_names,
        )
        mlflow.log_figure(figure=importances, artifact_file="feature_importances.png")

In [8]:
hyperparameter_tuning(
    X_train=X_train_tf, y_train=y_train, feat_names=pp.get_feature_names_out()
)

[I 2025-09-15 15:03:44,725] A new study created in memory with name: no-name-ac25ba23-365e-46a5-aaab-d5bd492f8c2a
/Users/hmnguyen1067/Downloads/Github/Bank-Customer-Churn-Prediction-Pipeline/.pixi/envs/default/lib/python3.12/site-packages/xgboost/callback.py:386: UserWarning: [15:03:44] WARNING: /Users/runner/miniforge3/conda-bld/xgboost-split_1755048541311/work/src/learner.cc:738: 
Parameters: { "n_estimators" } are not used.

  self.starting_round = model.num_boosted_rounds()
[I 2025-09-15 15:03:44,791] Trial 0 finished with value: 0.7070893684106809 and parameters: {'n_estimators': 1935, 'learning_rate': 0.7969454818643928, 'reg_lambda': 0.11270245072599183, 'reg_alpha': 0.0038480732119896897, 'subsample': 0.24041677639819287, 'max_depth': 2, 'max_delta_step': 0, 'min_child_weight': 9, 'gamma': 0.000169465562039471, 'scale_pos_weight': 1.4437836359206417}. Best is trial 0 with value: 0.7070893684106809.
/Users/hmnguyen1067/Downloads/Github/Bank-Customer-Churn-Prediction-Pipeline/.pi

🏃 View run delicate-fish-980 at: http://localhost:5000/#/experiments/1/runs/28fb241779b3495b949eec8bf585727a
🧪 View experiment at: http://localhost:5000/#/experiments/1
🏃 View run sincere-dog-180 at: http://localhost:5000/#/experiments/1/runs/377a0cb1f04a42ce87fa11e5d8dfdeb1
🧪 View experiment at: http://localhost:5000/#/experiments/1
🏃 View run illustrious-crow-941 at: http://localhost:5000/#/experiments/1/runs/67713189069d4e4fbc01cc464e1b6883
🧪 View experiment at: http://localhost:5000/#/experiments/1


/Users/hmnguyen1067/Downloads/Github/Bank-Customer-Churn-Prediction-Pipeline/.pixi/envs/default/lib/python3.12/site-packages/xgboost/callback.py:386: UserWarning: [15:03:44] WARNING: /Users/runner/miniforge3/conda-bld/xgboost-split_1755048541311/work/src/learner.cc:738: 
Parameters: { "n_estimators" } are not used.

  self.starting_round = model.num_boosted_rounds()
[I 2025-09-15 15:03:44,943] Trial 3 finished with value: 0.5 and parameters: {'n_estimators': 3077, 'learning_rate': 0.021930485556643693, 'reg_lambda': 5.194784342189012e-09, 'reg_alpha': 27.399390988843805, 'subsample': 0.9690688297671034, 'max_depth': 10, 'max_delta_step': 3, 'min_child_weight': 1, 'gamma': 0.000895617505524074, 'scale_pos_weight': 0.006743313339480342}. Best is trial 0 with value: 0.7070893684106809.


🏃 View run abrasive-deer-517 at: http://localhost:5000/#/experiments/1/runs/a13795eb006446a690d9b18db1dcb1b1
🧪 View experiment at: http://localhost:5000/#/experiments/1


/Users/hmnguyen1067/Downloads/Github/Bank-Customer-Churn-Prediction-Pipeline/.pixi/envs/default/lib/python3.12/site-packages/xgboost/callback.py:386: UserWarning: [15:03:44] WARNING: /Users/runner/miniforge3/conda-bld/xgboost-split_1755048541311/work/src/learner.cc:738: 
Parameters: { "n_estimators" } are not used.

  self.starting_round = model.num_boosted_rounds()
[I 2025-09-15 15:03:44,994] Trial 4 finished with value: 0.5 and parameters: {'n_estimators': 698, 'learning_rate': 0.09780337016659407, 'reg_lambda': 2.3893167752763565e-09, 'reg_alpha': 10.058296249730986, 'subsample': 0.33290198344001526, 'max_depth': 8, 'max_delta_step': 3, 'min_child_weight': 6, 'gamma': 5.699231800614222e-05, 'scale_pos_weight': 4.0554902701197996e-05}. Best is trial 0 with value: 0.7070893684106809.
/Users/hmnguyen1067/Downloads/Github/Bank-Customer-Churn-Prediction-Pipeline/.pixi/envs/default/lib/python3.12/site-packages/xgboost/callback.py:386: UserWarning: [15:03:45] WARNING: /Users/runner/minifor

...

🏃 View run efficient-snake-666 at: http://localhost:5000/#/experiments/1/runs/927c6f11be6d46cf9ff3141266efbb20
🧪 View experiment at: http://localhost:5000/#/experiments/1
🏃 View run upbeat-midge-514 at: http://localhost:5000/#/experiments/1/runs/01f1a62a370549dabccec2bc41f6b208
🧪 View experiment at: http://localhost:5000/#/experiments/1
🏃 View run delicate-eel-479 at: http://localhost:5000/#/experiments/1/runs/3fec01996aa1419ca1f0f8d357a98961
🧪 View experiment at: http://localhost:5000/#/experiments/1


/Users/hmnguyen1067/Downloads/Github/Bank-Customer-Churn-Prediction-Pipeline/.pixi/envs/default/lib/python3.12/site-packages/xgboost/callback.py:386: UserWarning: [15:03:58] WARNING: /Users/runner/miniforge3/conda-bld/xgboost-split_1755048541311/work/src/learner.cc:738: 
Parameters: { "n_estimators" } are not used.

  self.starting_round = model.num_boosted_rounds()
[I 2025-09-15 15:03:58,947] Trial 198 finished with value: 0.750036949453148 and parameters: {'n_estimators': 1093, 'learning_rate': 0.5649452141073311, 'reg_lambda': 9.925603573074257, 'reg_alpha': 3.5633815064369796e-07, 'subsample': 0.34065495987200867, 'max_depth': 3, 'max_delta_step': 10, 'min_child_weight': 8, 'gamma': 0.07144509275241318, 'scale_pos_weight': 8.721696924563222}. Best is trial 163 with value: 0.7955709922159819.
/Users/hmnguyen1067/Downloads/Github/Bank-Customer-Churn-Prediction-Pipeline/.pixi/envs/default/lib/python3.12/site-packages/xgboost/callback.py:386: UserWarning: [15:03:58] WARNING: /Users/run

🏃 View run ambitious-chimp-293 at: http://localhost:5000/#/experiments/1/runs/20af76f16ef342ab8cfb0cd44f046bee
🧪 View experiment at: http://localhost:5000/#/experiments/1
🏃 View run enthused-sponge-849 at: http://localhost:5000/#/experiments/1/runs/f50adcb195684edfb3689073b2c2c0a9
🧪 View experiment at: http://localhost:5000/#/experiments/1
Number of finished trials: 200
Best value: 0.7955709922159819


2025/09/15 15:04:00 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
Successfully registered model 'XGBoostChurnModel'.
2025/09/15 15:04:00 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: XGBoostChurnModel, version 1
Created version '1' of model 'XGBoostChurnModel'.


🏃 View run xgboost_hyperparameter_tuning_2025-09-15 at: http://localhost:5000/#/experiments/1/runs/7b38e29d23c948b6946da801ba6739fa
🧪 View experiment at: http://localhost:5000/#/experiments/1


In [10]:
loaded_preprocessor = mlflow.sklearn.load_model("models:/ChurnDataPreprocessor/latest")
loaded_model = mlflow.xgboost.load_model("models:/XGBoostChurnModel/latest")

In [11]:
def make_predictions(
    model, X_test: pd.DataFrame, preprocessor: ColumnTransformer
) -> np.ndarray:
    X_test = preprocessor.transform(X_test)
    preds = model.predict(xgb.DMatrix(X_test))
    return np.clip(np.rint(preds), 0, 1).astype(int)

In [12]:
preds = make_predictions(loaded_model, X_test, loaded_preprocessor)

In [13]:
preds_tr = make_predictions(loaded_model, X_train, loaded_preprocessor)
X_train = X_train.assign(Preds=preds_tr)
preds_val = make_predictions(loaded_model, X_val, loaded_preprocessor)
X_val = X_val.assign(Preds=preds_val)
preds_t = make_predictions(loaded_model, X_test, loaded_preprocessor)
X_test = X_test.assign(Preds=preds_t)

In [14]:
X_trv = pd.concat([X_train, X_val], axis=0)
X_trv

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Complain,Satisfaction Score,Card Type,Point Earned,Preds
4791,4792,15746461,Taylor,709,Spain,Male,35,2,0.00,2,1,0,104982.39,0,2,GOLD,422,0
8881,8882,15618647,Kornilova,744,France,Male,29,1,43504.42,1,1,1,119327.75,0,1,PLATINUM,607,0
6166,6167,15567431,Kodilinyechukwu,773,France,Male,64,2,145578.28,1,0,1,186172.85,0,1,SILVER,630,1
4473,4474,15713532,Wang,646,Germany,Female,29,4,105957.44,1,1,0,15470.91,0,1,PLATINUM,345,1
854,855,15601589,Baresi,675,France,Female,57,8,0.00,2,0,1,95463.29,0,3,SILVER,632,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5392,5393,15710012,Bowen,738,Spain,Male,44,2,0.00,2,1,0,43018.82,1,5,PLATINUM,546,1
2328,2329,15664204,Meany,706,Spain,Male,29,2,0.00,2,1,1,18255.51,0,1,SILVER,868,0
6826,6827,15727361,Chiemela,547,France,Female,51,1,0.00,2,1,1,56908.41,0,2,SILVER,868,0
7511,7512,15686913,Kung,757,France,Male,38,0,0.00,1,1,0,83263.06,0,1,PLATINUM,537,1


# Evidently Report

In [15]:
from evidently import Report, DataDefinition, Dataset
from evidently.presets import DataDriftPreset
from evidently.ui.workspace import RemoteWorkspace
from evidently.metrics import ValueDrift, DriftedColumnsCount, MissingValueCount

ws = RemoteWorkspace("http://localhost:8000")

In [22]:
if proj_list := ws.search_project("Churn Prediction Project"):
    proj_id = proj_list[0].id
    project = ws.get_project(proj_id)
else:
    project = ws.create_project(name="Churn Prediction Project")

In [23]:
data_definition = DataDefinition(
    numerical_columns=num,
    categorical_columns=cat + ["Exited", "Preds"],
)

cur_data = Dataset.from_pandas(
    data=X_train.assign(Exited=y_train),
    data_definition=data_definition,
)

ref_data = Dataset.from_pandas(
    data=X_test.assign(Exited=y_test),
    data_definition=data_definition,
)

report = Report(
    [
        ValueDrift(column="Preds"),
        DriftedColumnsCount(),
        MissingValueCount(column="Preds"),
        DataDriftPreset(),
    ],
    include_tests=True,
)

eval = report.run(cur_data, reference_data=ref_data)

In [24]:
ws.add_run(project.id, eval)

Report ID: 01994c69-c77a-7a05-ba8f-b469d6d1765c
Link: http://localhost:8000/projects/01994c69-198e-72ab-b608-3cab5dcf4af5/reports/01994c69-c77a-7a05-ba8f-b469d6d1765c

In [25]:
eval.dict()["metrics"]

[{'id': 'f989471f535449d947dc1594bee11bbe',
  'metric_id': 'ValueDrift(column=Preds)',
  'value': np.float64(0.07368249443448782)},
 {'id': '15e89f895b482f9b84ba7274ed18a106',
  'metric_id': 'DriftedColumnsCount(drift_share=0.5)',
  'value': {'count': 3.0, 'share': 0.2}},
 {'id': '4c3d37d67814b7823c164fbf505544bc',
  'metric_id': 'MissingValueCount(column=Preds)',
  'value': {'count': 0.0, 'share': np.float64(0.0)}},
 {'id': 'c4f1631539c3955b53e20cab2fc9e838',
  'metric_id': 'ValueDrift(column=CreditScore)',
  'value': np.float64(0.9663582551858991)},
 {'id': '8f5d1c60a32d6fc1bd54bc53af61d8e8',
  'metric_id': 'ValueDrift(column=Age)',
  'value': np.float64(0.4592764651990313)},
 {'id': '44af7ff50b9319ad30493a39cabd4056',
  'metric_id': 'ValueDrift(column=Tenure)',
  'value': np.float64(0.9830135526838559)},
 {'id': 'f11fe99e16eef1d959d624000c604160',
  'metric_id': 'ValueDrift(column=Balance)',
  'value': np.float64(0.4936461401575463)},
 {'id': '40bb5b6d43598f6e48fe609e4d9cb46c',
  'm

In [26]:
results = eval.dict()
prediction_drift = results["metrics"][0]["value"]
num_drifted_columns = results["metrics"][1]["value"]["count"]
share_missing_values = results["metrics"][2]["value"]["share"]

In [28]:
CONNECTION_STRING = "host=localhost port=5434 user=grafana password=grafana"
CONNECTION_STRING_DB = CONNECTION_STRING + " dbname=grafana"

create_table_statement = """
drop table if exists metrics;
create table metrics(
	timestamp timestamp,
	prediction_drift float,
	num_drifted_columns integer,
	share_missing_values float
)
"""

In [30]:
import psycopg

with psycopg.connect(CONNECTION_STRING, autocommit=True) as conn:
    res = conn.execute("SELECT 1 FROM pg_database WHERE datname='grafana'")
    if len(res.fetchall()) == 0:
        conn.execute("create database grafana;")
    with psycopg.connect(CONNECTION_STRING_DB) as conn:
        conn.execute(create_table_statement)

In [36]:
with psycopg.connect(CONNECTION_STRING_DB, autocommit=True) as conn:
    with conn.cursor() as curr:
        curr.execute(
            "insert into metrics(timestamp, prediction_drift, num_drifted_columns, share_missing_values) values (%s, %s, %s, %s)",
            (
                datetime.date.today(),
                prediction_drift,
                num_drifted_columns,
                share_missing_values,
            ),
        )